## 1: Environment Setup & Configuration

In [ ]:
!pip install -q iterative-stratification
!pip install -q segmentation-models-pytorch

In [ ]:
import os
import gc
import cv2
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from tqdm import tqdm

# Kaggle utils
import kagglehub
from contextlib import nullcontext

# Set seed for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [ ]:
# Hardware acceleration setup
def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    elif torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

device = get_device()
num_gpus = 0

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    num_gpus = torch.cuda.device_count() if device.type == "cuda" else 0

print(f"Device: {device}")
print(f"Available GPUs: {num_gpus}")

# Paths & Global Constants
ROOT = Path("/kaggle/input/competitions/industrial-surface-defect-detection/public/")
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"
CSV_PATH = ROOT / "train.csv"
SAMPLE_SUB_PATH = ROOT / "sample_submission.csv"

In [ ]:
DEFECT_TYPES = ["defect_a", "defect_b", "defect_c", "defect_d"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(DEFECT_TYPES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

# Configuration Hyperparameters
CONFIG = {
    "seed": 42, "lr": 1e-3, "weight_decay": 1e-4, "num_workers": 4,

    "image_channels": 1, "num_classes": len(DEFECT_TYPES),

    # Prediction settings
    "threshold": 0.50,  # Fallback global threshold
    "class_thresholds": [0.25, 0.25, 0.60, 0.50],
    "min_defect_pixels": 255,
    "min_component_pixels": 75,

    # Image dimensions
    "image_height": 128, "image_width": 800,
}

In [ ]:
# Handle training
SUBMISSION = True

In [ ]:
# wandb login skipped: inference/sweep mode (no training)


## 2: Exploratory Data Analysis (EDA) & Dataset Processing

In [ ]:
df = pd.read_csv(CSV_PATH)

# Build unique image metadata DataFrame preserving appearance order
image_df = pd.DataFrame({"image_id": df["image_id"].drop_duplicates().values})

# Initialize binary defect targets
for defect in DEFECT_TYPES:
    image_df[defect] = 0

# Annotate defect presence per image
for defect in DEFECT_TYPES:
    positive_ids = df.loc[
        (df["defect_type"] == defect) & 
        (df["mask_rle"].notna()) & 
        (df["mask_rle"].astype(str).str.strip() != ""), 
        "image_id"
    ].unique()
    image_df.loc[image_df["image_id"].isin(positive_ids), defect] = 1

# Calculate defect statistics
image_df["total_defects"] = image_df[DEFECT_TYPES].sum(axis=1)

# Display tabular sample
print("Dataset Summary:")
print(f"Total Unique Images: {len(image_df)}")
display(image_df.head(5))

# Plot defect distribution analytics
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Class frequency distribution
defect_counts = image_df[DEFECT_TYPES].sum().reset_index()
defect_counts.columns = ["Defect Type", "Count"]
sns.barplot( data=defect_counts, x="Defect Type", y="Count", hue="Defect Type", ax=axes[0], palette="viridis", legend=False )
axes[0].set_title("Defect Instance Counts Across Dataset", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Positive Images")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', xytext=(0, 3), textcoords='offset points')

# Plot 2: Defect multiplicity per image
multi_counts = image_df["total_defects"].value_counts().reset_index()
multi_counts.columns = ["Defects Per Image", "Count"]
sns.barplot( data=multi_counts, x="Defects Per Image", y="Count", hue="Defects Per Image", ax=axes[1], palette="magma", legend=False )
axes[1].set_title("Defect Multiplicity per Image (0 to 4)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Number of Images")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', xytext=(0, 3), textcoords='offset points')

plt.tight_layout()
plt.show()

## 3: Run-Length Encoding (RLE) Utilities & Validation

In [ ]:
def rle_decode(rle_string, height, width):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle_string) or not str(rle_string).strip():
        return mask.reshape((height, width), order="F")

    rle = np.asarray(str(rle_string).split(), dtype=np.int64)
    starts = rle[0::2] - 1
    lengths = rle[1::2]
    ends = starts + lengths

    if len(starts) != len(lengths):
        raise ValueError("Invalid RLE pairing.")

    for start, end in zip(starts, ends):
        mask[start:end] = 1

    return mask.reshape((height, width), order="F")


def rle_encode(mask):
    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("Mask must be a 2D array")

    pixels = (mask > 0).flatten(order="F")

    padded = np.concatenate([[0], pixels, [0]])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1

    changes[1::2] -= changes[::2]

    return " ".join(map(str, changes))

## 4: Sample Visualization (Train & Test Images)

In [ ]:
def visualize_sample_defects(df, annotations_df, image_dir, num_samples=3):
    # Sample images containing defects
    sample_ids = df[df["total_defects"] > 0]["image_id"].sample(num_samples, random_state=42).values
    colors = [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 0)] # R, G, B, Yellow

    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 2 * num_samples))
    
    for idx, img_id in enumerate(sample_ids):
        image_name = str(img_id) if Path(str(img_id)).suffix != "" else f"{img_id}.jpg"
        img_path = Path(image_dir) / image_name
        image = np.asarray(Image.open(img_path).convert("L"))
        h, w = image.shape
        
        # Build color overlay mask
        overlay = np.zeros((h, w, 3), dtype=np.float32)
        img_annotations = annotations_df[annotations_df["image_id"] == img_id]
        
        active_defects = []
        for _, row in img_annotations.iterrows():
            defect_type = row["defect_type"]
            c_idx = CLASS_TO_IDX[defect_type]
            mask = rle_decode(row["mask_rle"], h, w)
            if mask.sum() > 0:
                active_defects.append(defect_type)
                for channel in range(3):
                    overlay[:, :, channel] += mask * colors[c_idx][channel]

        overlay = np.clip(overlay, 0, 1)

        # Original Image
        axes[idx, 0].imshow(image, cmap="gray")
        axes[idx, 0].set_title(f"Image ID: {img_id} (Original Grayscale)")
        axes[idx, 0].axis("off")

        # Composite Overlay
        axes[idx, 1].imshow(image, cmap="gray")
        axes[idx, 1].imshow(overlay, alpha=0.5)
        axes[idx, 1].set_title(f"Defect Overlay: {', '.join(active_defects) if active_defects else 'None'}")
        axes[idx, 1].axis("off")

    plt.tight_layout()
    plt.show()

# Render sample visualizations
visualize_sample_defects(image_df, df, TRAIN_DIR, num_samples=3)

## 5: Data Splitting & Memory-Efficient PyTorch Dataset

### 5.1 Stratified Multi-label Train-Validation Split

In [ ]:
# Prepare features and multi-label targets
X = image_df["image_id"].values
y = image_df[DEFECT_TYPES].values

# Perform stratified split
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=CONFIG["seed"])
train_idx, valid_idx = next(splitter.split(X, y))

# Create train and validation DataFrames
train_df = image_df.iloc[train_idx].reset_index(drop=True)
valid_df = image_df.iloc[valid_idx].reset_index(drop=True)

# Split annotations
train_ids_set = set(train_df["image_id"])
valid_ids_set = set(valid_df["image_id"])

train_annotations = df[df["image_id"].isin(train_ids_set)].copy()
valid_annotations = df[df["image_id"].isin(valid_ids_set)].copy()

# Display split information
print(f"Train samples      : {len(train_df):,}")
print(f"Validation samples : {len(valid_df):,}")
print(f"Train annotations  : {len(train_annotations):,}")
print(f"Valid annotations  : {len(valid_annotations):,}")

### 5.2 Preload Images and Generate Masks

In [ ]:
def preload_data(image_df, annotations_df=None, image_dir=TRAIN_DIR, desc=None, is_test=False):
    image_ids = image_df["image_id"].values
    image_dir = Path(image_dir)
    n = len(image_ids)

    # Get image dimensions
    if not is_test:
        height, width = int(annotations_df["height"].iloc[0]), int(annotations_df["width"].iloc[0])
    else:
        height, width = int(df["height"].iloc[0]), int(df["width"].iloc[0])

    # Allocate float32 arrays
    images = np.empty((n, 1, height, width), dtype=np.float32)

    if not is_test:
        masks = np.zeros((n, CONFIG["num_classes"], height, width), dtype=np.float32)
        annotation_groups = {str(img_id): group for img_id, group in annotations_df.groupby("image_id")}

    # Load images and masks
    for idx, image_id in enumerate(tqdm(image_ids, desc=f"Loading {desc}", unit="img", total=n, dynamic_ncols=True)):
        img_id = str(image_id)
        img_name = img_id if Path(img_id).suffix else f"{img_id}.jpg"
        img_path = image_dir / img_name

        # Load and normalize image
        with Image.open(img_path) as img:
            image = np.asarray(img.convert("L"), dtype=np.float32)

        if image.shape != (height, width):
            raise ValueError(f"Image {img_id} has unexpected shape {image.shape}")

        images[idx, 0] = image / 255.0

        # Generate masks
        if not is_test:
            img_annotations = annotation_groups.get(img_id)

            if img_annotations is not None:
                for row in img_annotations.itertuples(index=False):
                    if pd.isna(row.mask_rle) or not str(row.mask_rle).strip():
                        continue

                    channel = CLASS_TO_IDX[row.defect_type]
                    decoded_mask = rle_decode(row.mask_rle, height, width)
                    masks[idx, channel] = np.maximum(masks[idx, channel], decoded_mask)

    if is_test:
        return images, image_ids

    return images, masks, image_ids

### 5.3 Custom PyTorch Dataset

In [ ]:
class SurfaceDefectDataset(Dataset):
    def __init__(self, images, masks=None, image_ids=None, is_test=False):
        self.images = torch.from_numpy(images)
        self.masks = torch.from_numpy(masks) if masks is not None else None
        self.image_ids, self.is_test = image_ids, is_test

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        if self.is_test:
            return image, str(self.image_ids[idx])
        return image, self.masks[idx]

### 3.4 Preload Data

In [ ]:
# Preload training and validation data
train_images, train_masks, train_ids = preload_data(image_df=train_df, annotations_df=train_annotations, image_dir=TRAIN_DIR, desc="Train")
valid_images, valid_masks, valid_ids = preload_data(image_df=valid_df, annotations_df=valid_annotations, image_dir=TRAIN_DIR, desc="Valid")
 
# Prepare and preload test data
sub_df = pd.read_csv(SAMPLE_SUB_PATH)
test_image_df = pd.DataFrame({"image_id": sub_df["image_id"].unique()})
    
test_images, test_ids = preload_data(image_df=test_image_df, image_dir=TEST_DIR, desc="Test", is_test=True)

### 5.5 Create PyTorch Datasets & DataLoaders

In [ ]:
def create_dataloaders(train_dataset, valid_dataset, test_dataset, batch_size=64, num_workers=2):
    
    device = get_device()
    pin_memory = device.type == "cuda"

    common_args = {"batch_size": batch_size, "num_workers": num_workers, "pin_memory": pin_memory, "persistent_workers": num_workers > 0}
    
    if num_workers > 0: common_args["prefetch_factor"] = 1

    train_loader = DataLoader(train_dataset, shuffle=True, **common_args)
    valid_loader = DataLoader(valid_dataset, shuffle=False, **common_args)
    test_loader = DataLoader(test_dataset, shuffle=False, **common_args)

    return train_loader, valid_loader, test_loader

In [ ]:
# Create datasets
train_dataset = SurfaceDefectDataset(images=train_images, masks=train_masks, image_ids=train_ids)
valid_dataset = SurfaceDefectDataset(images=valid_images, masks=valid_masks, image_ids=valid_ids)
test_dataset = SurfaceDefectDataset(images=test_images, image_ids=test_ids, is_test=True)

# Create DataLoaders
BatchSize = 8
num_workers = 2

train_loader, valid_loader, test_loader = create_dataloaders( train_dataset, valid_dataset, test_dataset, batch_size=BatchSize, num_workers=num_workers) 

# Dataset and DataLoader summary
print(f"{'Dataset':<12} | {'Samples':>10} | {'Batches':>10}")
print("-" * 39)

for name, dataset, loader in [ ("Train", train_dataset, train_loader), ("Validation", valid_dataset, valid_loader), ("Test", test_dataset, test_loader) ]:
    print(f"{name:<12} | {len(dataset):>10,} | {len(loader):>10,}")

print("-" * 39)
print(f"Batch Size: {BatchSize} | DataLoaders created successfully!")

images, masks = next(iter(train_loader))

print("\nImages shape:", images.shape)
print("Masks shape :", masks.shape)

## 6: CNN-Model Architecture

### 6.1: Model 1:- Scratch UNET Like Architecture

In [ ]:
# 1. Double Convolution Block
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


# 2. Encoder Block
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.down = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.down(x)


# 3. Decoder Block
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))


# 4. Baseline U-Net
class DefectUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=4):
        super().__init__()

        self.encoder = nn.Sequential(
            DoubleConv(in_channels, 32), DownBlock(32, 64), 
            DownBlock(64, 128), DownBlock(128, 256) )

        self.bottleneck = nn.Sequential( nn.MaxPool2d(2), DoubleConv(256, 512) )

        self.decoder = nn.ModuleList([
            UpBlock(512, 256, 256), UpBlock(256, 128, 128),
            UpBlock(128, 64, 64), UpBlock(64, 32, 32) ])

        self.outc = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        skips = []

        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        x = self.bottleneck(x)

        for layer, skip in zip(self.decoder, reversed(skips)):
            x = layer(x, skip)

        return self.outc(x)

In [ ]:
# 1. Residual Convolution Block
class ResidualConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

        # Match input channels with output channels
        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv(x)

        return self.relu(out + identity)


# 2. Encoder Block
class ResidualDownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.down = nn.Sequential(
            nn.MaxPool2d(2),
            ResidualConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.down(x)


# 3. Decoder Block
class ResidualUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=2, stride=2
        )

        self.conv = ResidualConv(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        # Handle odd image dimensions
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat([x, skip], dim=1)

        return self.conv(x)


# 4. Residual U-Net
class ResidualDefectUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=4):
        super().__init__()

        self.encoder = nn.ModuleList([
            ResidualConv(in_channels, 32),
            ResidualDownBlock(32, 64),
            ResidualDownBlock(64, 128),
            ResidualDownBlock(128, 256)
        ])

        self.bottleneck = nn.Sequential(
            nn.MaxPool2d(2),
            ResidualConv(256, 512)
        )

        self.decoder = nn.ModuleList([
            ResidualUpBlock(512, 256, 256),
            ResidualUpBlock(256, 128, 128),
            ResidualUpBlock(128, 64, 64),
            ResidualUpBlock(64, 32, 32)
        ])

        self.outc = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):

        skips = []

        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        x = self.bottleneck(x)

        for layer, skip in zip(self.decoder, reversed(skips)):
            x = layer(x, skip)

        return self.outc(x)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, gating_channels, skip_channels, inter_channels):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(gating_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.W_x = nn.Sequential(nn.Conv2d(skip_channels, inter_channels, 1, bias=False), nn.BatchNorm2d(inter_channels))
        self.psi = nn.Sequential(nn.Conv2d(inter_channels, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        attention = self.relu(g1 + x1)
        attention = self.psi(attention)
        return x * attention


class AttentionUpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.attention = AttentionGate(gating_channels=out_channels, skip_channels=skip_channels, inter_channels=max(out_channels // 2, 1))
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        # Match spatial dimensions
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)

        # Apply attention to encoder features
        skip = self.attention(x, skip)

        # Concatenate features
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class AttentionDefectUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=4):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            DoubleConv(in_channels, 32),
            DownBlock(32, 64),
            DownBlock(64, 128),
            DownBlock(128, 256)
        )

        # Bottleneck
        self.bottleneck = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))

        # Attention Decoder
        self.decoder = nn.ModuleList([
            AttentionUpBlock(512, 256, 256),
            AttentionUpBlock(256, 128, 128),
            AttentionUpBlock(128, 64, 64),
            AttentionUpBlock(64, 32, 32)
        ])

        # Output
        self.outc = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        skips = []
        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        x = self.bottleneck(x)

        for layer, skip in zip(self.decoder, reversed(skips)):
            x = layer(x, skip)

        return self.outc(x)

### 6.2: Model 2:- U-Net + ResNet34

In [ ]:
def get_resnet34_unet():

    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",

        in_channels=1,
        classes=4,

        activation=None
    )

    return model

### 6.3: Model 2: U-Net + EfficientNet-B0

In [ ]:
def get_efficientnet_b0_unet():

    model = smp.Unet(
        encoder_name="efficientnet-b0",
        encoder_weights="imagenet",

        in_channels=1,
        classes=4,

        activation=None
    )

    return model

## 7: Model Training

### 7.1: Custom Loss Function and Metrics

In [ ]:
class SoftDiceBCELoss(nn.Module):
    def __init__(self, bce_weight=0.4, dice_weight=0.6, smooth=1e-7):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        
        probs = torch.sigmoid(logits)
        probs_flat = probs.flatten(start_dim=2)
        targets_flat = targets.flatten(start_dim=2)
        
        intersection = (probs_flat * targets_flat).sum(dim=2)
        cardinality = probs_flat.sum(dim=2) + targets_flat.sum(dim=2)
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss = 1.0 - dice_score.mean()
        
        return (self.bce_weight * bce_loss) + (self.dice_weight * dice_loss)


@torch.no_grad()
def calculate_competition_metrics(predictions, targets, threshold=None, min_defect_pixels=None, eps=1e-7):
    if threshold is None:
        threshold = CONFIG["class_thresholds"]
    if min_defect_pixels is None:
        min_defect_pixels = CONFIG["min_defect_pixels"]

    # Convert thresholds to tensor
    if isinstance(threshold, (list, tuple)):
        threshold = torch.tensor(threshold, device=predictions.device, dtype=predictions.dtype).view(1, -1, 1, 1)

    probs = torch.sigmoid(predictions)
    pred_masks = (probs >= threshold).float()
    targets = (targets > 0.5).float()

    # Minimum pixel filtering
    pred_pixels = pred_masks.sum(dim=(2, 3), keepdim=True)
    pred_masks = torch.where(pred_pixels >= min_defect_pixels, pred_masks, torch.zeros_like(pred_masks))

    pred_flat = pred_masks.flatten(start_dim=2)
    target_flat = targets.flatten(start_dim=2)

    pred_pixels = pred_flat.sum(dim=2)
    target_pixels = target_flat.sum(dim=2)
    intersection = (pred_flat * target_flat).sum(dim=2)
    total_pixels = targets.shape[-2] * targets.shape[-1]

    positive_pairs = target_pixels > 0
    empty_pairs = ~positive_pairs

    dice_per_pair = (2 * intersection + eps) / (pred_pixels + target_pixels + eps)
    foreground_dice = dice_per_pair[positive_pairs].mean().item() if positive_pairs.any() else 0.0

    fp_fraction = pred_pixels / total_pixels
    fp_score = 1.0 - fp_fraction[empty_pairs].mean().item() if empty_pairs.any() else 1.0

    correctly_empty = pred_pixels == 0
    empty_accuracy = correctly_empty[empty_pairs].float().mean().item() if empty_pairs.any() else 1.0

    score = 0.70 * foreground_dice + 0.20 * fp_score + 0.10 * empty_accuracy

    return {"score": score, "foreground_dice": foreground_dice, "false_positive_score": fp_score, "empty_accuracy": empty_accuracy}

@torch.no_grad()
def calculate_filtered_metrics(pred_masks, targets, eps=1e-7):
    pred_masks = pred_masks.float().cpu()
    targets = (targets.cpu() > 0.5).float()

    pred_flat = pred_masks.flatten(start_dim=2)
    target_flat = targets.flatten(start_dim=2)

    pred_pixels = pred_flat.sum(dim=2)
    target_pixels = target_flat.sum(dim=2)
    intersection = (pred_flat * target_flat).sum(dim=2)
    total_pixels = targets.shape[-2] * targets.shape[-1]

    positive_pairs = target_pixels > 0
    empty_pairs = ~positive_pairs

    dice_per_pair = (2 * intersection + eps) / (pred_pixels + target_pixels + eps)
    foreground_dice = dice_per_pair[positive_pairs].mean().item() if positive_pairs.any() else 0.0

    fp_fraction = pred_pixels / total_pixels
    fp_score = 1.0 - fp_fraction[empty_pairs].mean().item() if empty_pairs.any() else 1.0

    correctly_empty = pred_pixels == 0
    empty_accuracy = correctly_empty[empty_pairs].float().mean().item() if empty_pairs.any() else 1.0

    score = 0.70 * foreground_dice + 0.20 * fp_score + 0.10 * empty_accuracy

    return {"score": score, "foreground_dice": foreground_dice, "false_positive_score": fp_score, "empty_accuracy": empty_accuracy}

### 7.2: Training Loop and Helper Functions

In [ ]:
# 1. Automatic Mixed Precision
def get_amp():
    if device.type == "cuda":
        scaler = torch.amp.GradScaler("cuda")
        autocast = lambda: torch.amp.autocast("cuda")
    else:
        scaler = None
        autocast = nullcontext
    return scaler, autocast


# 2. Training
def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device, log_every=1, desc="Training"):
    model.train()
    total_loss, total_samples = 0.0, 0

    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(dataloader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for batch_idx, (images, targets) in enumerate(pbar, start=1):
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        batch_size = images.size(0)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

        total_loss += loss.detach().item() * batch_size
        total_samples += batch_size

        batch_metrics = calculate_competition_metrics(logits.detach(), targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        if batch_idx % log_every == 0 or batch_idx == len(dataloader):
            avg_loss = total_loss / total_samples
            avg_metrics = {k: v / metric_count for k, v in metrics_sum.items()}

            pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}", 
                               "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}" })

        del logits, loss

    epoch_metrics = {k: v / metric_count for k, v in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics

def apply_prediction_filters(probs):
    thresholds = torch.tensor(CONFIG["class_thresholds"], device=probs.device).view(1, -1, 1, 1)

    # Convert probabilities into binary masks
    masks = (probs >= thresholds).byte().cpu().numpy()
    filtered_masks = np.zeros_like(masks, dtype=np.uint8)

    min_component = CONFIG["min_component_pixels"]
    min_pixels = CONFIG["min_defect_pixels"]

    for b in range(masks.shape[0]):
        for c in range(masks.shape[1]):
            mask = masks[b, c]

            # Connected component filtering
            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
            filtered = np.zeros_like(mask)

            for i in range(1, num_labels):
                area = stats[i, cv2.CC_STAT_AREA]
                if area >= min_component:
                    filtered[labels == i] = 1

            # Whole-mask minimum pixel filtering
            if filtered.sum() >= min_pixels:
                filtered_masks[b, c] = filtered

    return torch.from_numpy(filtered_masks).to(probs.device)
    
@torch.inference_mode()
def predict(model, images, device):
    
    model.eval()
    images = images.to(device, non_blocking=True)

    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16): logits = model(images)
            
    else: logits = model(images)
    return logits

# 3. Validation
@torch.inference_mode()
def validate_epoch(model, dataloader, criterion, device, log_every=1, desc="Validation"):
    
    model.eval()
    total_loss, total_samples = 0.0, 0

    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(dataloader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for batch_idx, (images, targets) in enumerate(pbar, start=1):
        targets = targets.to(device, non_blocking=True)
        batch_size = images.size(0)

        logits = predict(model, images, device)
        loss = criterion(logits, targets)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        batch_metrics = calculate_competition_metrics(logits, targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        if batch_idx % log_every == 0 or batch_idx == len(dataloader):
            avg_loss = total_loss / total_samples
            avg_metrics = {k: v / metric_count for k, v in metrics_sum.items()}

            pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}",
                "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}" })

        del logits, loss

    epoch_metrics = {k: v / metric_count for k, v in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics


@torch.inference_mode()
def validate_epoch_filtered(model, loader, criterion, device, desc="Filtered Validation"):
    model.eval()
    total_loss, total_samples = 0.0, 0
    metrics_sum = {"score": 0.0, "foreground_dice": 0.0, "false_positive_score": 0.0, "empty_accuracy": 0.0}
    metric_count = 0

    pbar = tqdm(loader, desc=desc, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {postfix}]")

    for images, targets in pbar:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        batch_size = images.size(0)

        logits = predict(model, images, device)

        # Calculate loss using original logits
        loss = criterion(logits, targets)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        # Apply prediction filters
        probs = torch.sigmoid(logits)
        pred_masks = apply_prediction_filters(probs)

        # Calculate filtered metrics
        batch_metrics = calculate_filtered_metrics(pred_masks, targets)

        for key in metrics_sum:
            metrics_sum[key] += batch_metrics[key]

        metric_count += 1

        # Calculate average metrics
        avg_loss = total_loss / total_samples
        avg_metrics = {key: value / metric_count for key, value in metrics_sum.items()}

        pbar.set_postfix({ "loss": f"{avg_loss:.4f}", "score": f"{avg_metrics['score']:.4f}", "dice": f"{avg_metrics['foreground_dice']:.4f}",
            "fp": f"{avg_metrics['false_positive_score']:.4f}", "empty": f"{avg_metrics['empty_accuracy']:.4f}"
        })

    epoch_metrics = {key: value / metric_count for key, value in metrics_sum.items()}
    epoch_metrics["loss"] = total_loss / total_samples

    return epoch_metrics

# 4. Complete Training Function
def fit_model(model, train_loader, valid_loader, optimizer, criterion, epochs, log_every=1, run_name=None, architecture="U-Net", save_path=None):
    
    if num_gpus > 1 and device.type == "cuda": model = torch.nn.DataParallel(model)
    scaler, autocast = get_amp()
    
    print(f"Starting Model Training.\nUsing Device: {device} | Available GPUs: {num_gpus}\n")

    # Initialize wandb
    run = wandb.init( entity="models-indian-institute-of-technology-madras8289", project="23f1000351-t32026", name=run_name ,
                      config={ "architecture": architecture, "epochs": epochs, "batch_size": train_loader.batch_size, "optimizer": optimizer.__class__.__name__, "learning_rate": optimizer.param_groups[0]["lr"], })
    
    run.watch(model, log="all", log_freq=7)

    metric_names = ["loss", "score", "foreground_dice", "false_positive_score", "empty_accuracy"]
    history = { "train": {name: [] for name in metric_names}, "valid": {name: [] for name in metric_names} }

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-7 )

    # Early stopping settings
    best_score = float("-inf")
    early_stop_counter = 0
    patience = 7
    min_delta = 0.0001

    for epoch in range(epochs):
        
        train_metrics = train_one_epoch( model, train_loader, optimizer, criterion, scaler, device, log_every=log_every, desc=f"[Train] [{epoch + 1}/{epochs}]" )
        # valid_metrics = validate_epoch( model, valid_loader, criterion, device, log_every=1, desc=f"[Valid] [{epoch + 1}/{epochs}]" )
        valid_metrics = validate_epoch_filtered( model, valid_loader, criterion, device, desc=f"[Valid] [{epoch + 1}/{epochs}]"  )

        for name in metric_names:
            history["train"][name].append(train_metrics[name])
            history["valid"][name].append(valid_metrics[name])

        wandb.log({ "trainLoss": train_metrics["loss"], "validLoss": valid_metrics["loss"], "trainScore": train_metrics["score"], "validScore": valid_metrics["score"],
                    "train_FgDice": train_metrics["foreground_dice"], "valid_FgDice": valid_metrics["foreground_dice"], 
                    "trainFalsePositive": train_metrics["false_positive_score"], "validFalsePositiveScore": valid_metrics["false_positive_score"], 
                    "trainEmptyAccuracy": train_metrics["empty_accuracy"], "validEmptyAccuracy": valid_metrics["empty_accuracy"], "epoch": epoch + 1 }, step=epoch + 1)
        
        # Update scheduler
        current_score = valid_metrics["score"]
        scheduler.step(current_score)

        # Save best model: architecture + weights
        if current_score > best_score + min_delta:
        
            best_score = current_score
            early_stop_counter = 0
        
            model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
            torch.save(model_to_save, save_path)
            print(f"Best model saved: {save_path}")
        
        else: early_stop_counter += 1
            
        print(f"LR: [{optimizer.param_groups[0]['lr']:.7f}] Early Stopping: [{early_stop_counter}/{patience}] Best Valid Score: [{best_score:.4f}]")

        gc.collect()

        # Stop training when validation score stops improving
        if early_stop_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break
        
    run.finish()
    return history

In [ ]:
# Plot Loss & Metric History
def plot_training_results(history):
    
    metrics = ["loss", "score", "foreground_dice", "false_positive_score", "empty_accuracy"]
    titles = ["Loss Convergence", "Competition Score", "Foreground Dice", "False Positive Score", "Empty Accuracy"]
    ylabels = ["Loss", "Score", "Dice", "Score", "Accuracy"]

    epochs = range(1, len(history["train"]["loss"]) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, (metric, title, ylabel) in enumerate(zip(metrics, titles, ylabels)):
        axes[i].plot(epochs, history["train"][metric], "o-", label=f"Train {metric}", linewidth=2)
        axes[i].plot(epochs, history["valid"][metric], "s-", label=f"Valid {metric}", linewidth=2)
        axes[i].set_title(title, fontsize=12, fontweight="bold")
        axes[i].set_xlabel("Epoch")
        axes[i].set_ylabel(ylabel)
        axes[i].grid(True, linestyle="--", alpha=0.6)
        axes[i].legend()

    axes[5].axis("off")
    plt.tight_layout()
    plt.show()

## 8: Training Models

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 8.1: Training Scratch UNET Model

In [ ]:
if not SUBMISSION:

    MODEL = "UNET"
    VARIATION = "BASELINE-V1.1"
    name = f"{MODEL}-{VARIATION}"
    run_name = f"{MODEL}-V1.1"

    # Initialize model, optimizer and loss
    defectunet_model = DefectUNet(in_channels=1, num_classes=4).to(device)
    save_path = f"/kaggle/working/{name}.pth"

    EPOCH = 64
    LR = CONFIG["lr"]
    WD = CONFIG["weight_decay"]

    optimizer = torch.optim.AdamW(defectunet_model.parameters(), lr=LR, weight_decay=WD)
    criterion = SoftDiceBCELoss(bce_weight=0.4, dice_weight=0.6)

    # Train model
    UNET_history = fit_model(model=defectunet_model, train_loader=train_loader, valid_loader=valid_loader, optimizer=optimizer, criterion=criterion, epochs=EPOCH, log_every=9, run_name=run_name, save_path=save_path)

    # Upload model to Kaggle
    handle = f"{KAGGLE_USERNAME}/{MODEL}/PyTorch/{VARIATION}"
    kagglehub.model_upload(handle, save_path, version_notes=f"{MODEL} trained best model: {VARIATION}")
    print(f"Model uploaded: {handle}")

    plot_training_results(UNET_history)
else:
    print("Submission/Testing mode activated. Skipping training!")

In [ ]:
if not SUBMISSION:

    VERSION = "V1.0"
    MODEL = "RESIDUAL-UNET"
    VARIATION = f"BASELINE-{VERSION}"

    name = f"{MODEL}-{VARIATION}"
    run_name = f"{MODEL}-{VERSION}"

    # Initialize Residual U-Net
    defectunet_model = ResidualDefectUNet( in_channels=CONFIG["image_channels"], num_classes=CONFIG["num_classes"] ).to(device)

    save_path = f"/kaggle/working/{name}.pth"

    EPOCH = 64
    LR = CONFIG["lr"]
    WD = CONFIG["weight_decay"]

    # Optimizer & Loss
    optimizer = torch.optim.AdamW( defectunet_model.parameters(), lr=LR, weight_decay=WD)
    criterion = SoftDiceBCELoss( bce_weight=0.4, dice_weight=0.6)

    # Train model
    RESUNET_history = fit_model( model=defectunet_model, train_loader=train_loader, valid_loader=valid_loader,
        optimizer=optimizer, criterion=criterion, epochs=EPOCH, log_every=9, run_name=run_name, architecture=MODEL, save_path=save_path
    )

    # Plot training results
    plot_training_results(RESUNET_history)

    # Upload model to Kaggle
    handle = f"{KAGGLE_USERNAME}/{MODEL}/PyTorch/{VARIATION}"
    kagglehub.model_upload( handle, save_path, version_notes=f"{MODEL} trained best model: {VARIATION}")
    print(f"Model uploaded: {handle}")

else:
    print("Submission/Testing mode activated. Skipping training!")

In [ ]:
if not SUBMISSION:
    VERSION = "V1.0"
    MODEL = "ATTENTION-UNET"
    VARIATION = f"BASELINE-{VERSION}"

    name = f"{MODEL}-{VARIATION}"
    run_name = f"{MODEL}-{VERSION}"

    # Initialize Attention U-Net
    attention_model = AttentionDefectUNet(in_channels=CONFIG["image_channels"], num_classes=CONFIG["num_classes"]).to(device)
    save_path = f"/kaggle/working/{name}.pth"

    EPOCH = 64
    LR = CONFIG["lr"]
    WD = CONFIG["weight_decay"]

    # Optimizer & Loss
    optimizer = torch.optim.AdamW(attention_model.parameters(), lr=LR, weight_decay=WD)
    criterion = SoftDiceBCELoss(bce_weight=0.4, dice_weight=0.6)

    # Train model
    ATTENTION_HISTORY = fit_model( model=attention_model, train_loader=train_loader, valid_loader=valid_loader,
        optimizer=optimizer, criterion=criterion, epochs=EPOCH, log_every=9,
        run_name=run_name, architecture=MODEL, save_path=save_path )

    # Upload model to Kaggle
    handle = f"{KAGGLE_USERNAME}/{MODEL}/PyTorch/{VARIATION}"
    kagglehub.model_upload(handle, save_path, version_notes=f"{MODEL} trained best model: {VARIATION}")
    print(f"Model uploaded: {handle}")

    # Plot training results
    plot_training_results(ATTENTION_HISTORY)

else:
    print("Submission/Testing mode activated. Skipping training!")

### 8.2: Fine Tune - Pretrained Model

In [ ]:
def get_optimizer(model, encoder_lr=1e-4, decoder_lr=1e-3, lr=1e-3, weight_decay=1e-4):

    optimizer = torch.optim.AdamW(
        [
            { "params": model.encoder.parameters(), "lr": encoder_lr },
            { "params": model.decoder.parameters(), "lr": decoder_lr },
            { "params": model.segmentation_head.parameters(), "lr": lr }
            
        ], weight_decay=weight_decay
    )

    return optimizer

criterion = SoftDiceBCELoss(bce_weight=0.4, dice_weight=0.6)

In [ ]:
if not SUBMISSION:

    VERSION = "V1.0"
    
    MODEL = "UNET-RESNET34"
    VARIATION = f"IMAGENET-FINETUNED-{VERSION}"
    
    name = f"{MODEL}-{VARIATION}"
    run_name = f"UNET-RESNET34-IMAGENET-{VERSION}"

    # Initialize model
    unetResNet34 = get_resnet34_unet().to(device)

    # Initialize optimizer & LOSS
    optimizer = get_optimizer( unetResNet34, lr=1e-3, encoder_lr=1e-4, weight_decay=CONFIG["weight_decay"])
    criterion = SoftDiceBCELoss( bce_weight=0.4, dice_weight=0.6)

    EPOCH = 32
    save_path = f"/kaggle/working/{name}.pth"

    # Train
    RESNET34_history = fit_model( model=unetResNet34, train_loader=train_loader, valid_loader=valid_loader,
        optimizer=optimizer, criterion=criterion, epochs=EPOCH, log_every=3, run_name=run_name,
        save_path=save_path, architecture="U-Net with ResNet34 ImageNet Encoder"
    )

    plot_training_results(RESNET34_history)

    # Upload to Kaggle
    handle = f"{KAGGLE_USERNAME}/{MODEL}/PyTorch/{VARIATION}"
    kagglehub.model_upload( handle, save_path, version_notes="U-Net with pretrained ResNet34 encoder" )
    print(f"Model uploaded: {handle}")
else:
    print("Submission/Testing mode activated. Skipping training!")

In [ ]:
if not SUBMISSION:

    VERSION = "V1.0"

    MODEL = "UNET-EFFICIENTNET-B0"
    VARIATION = f"IMAGENET-FINETUNED-{VERSION}"

    name = f"{MODEL}-{VARIATION}"
    run_name = f"EFFICIENTNET-B0-UNET-IMAGENET-{VERSION}"

    # Initialize model, optimizer & Loss
    efficientNetB0_Unet = get_efficientnet_b0_unet().to(device)
    
    optimizer = get_optimizer( efficientNetB0_Unet, lr=1e-3, encoder_lr=1e-4, weight_decay=CONFIG["weight_decay"] )
    criterion = SoftDiceBCELoss( bce_weight=0.4, dice_weight=0.6 )

    EPOCH = 32
    save_path = f"/kaggle/working/{name}.pth"

    # Train
    EFFICIENTNET_history = fit_model( model=efficientNetB0_Unet, train_loader=train_loader, valid_loader=valid_loader,
        optimizer=optimizer, criterion=criterion, epochs=EPOCH, log_every=9, run_name=run_name,
        save_path=save_path, architecture="U-Net with EfficientNet-B0 ImageNet Encoder" )

    plot_training_results(EFFICIENTNET_history)

    # Upload to Kaggle
    handle = f"{KAGGLE_USERNAME}/{MODEL}/PyTorch/{VARIATION}"
    kagglehub.model_upload( handle, save_path, version_notes="U-Net with pretrained EfficientNet-B0 encoder")
    print(f"Model uploaded: {handle}")
else:
    print("Submission/Testing mode activated. Skipping training!")

## Downloading Uploaded Model For Inference

In [ ]:
def load_attached_model(device):
    model_files = list(Path("/kaggle/input").rglob("*.pth"))
    if not model_files: raise FileNotFoundError("No .pth model attached!")
    model_path = model_files[0]
    model = torch.load(model_path, map_location=device, weights_only=False)
    model = model.to(device).eval()
    print(f"Model loaded: {model_path}")
    return model


## Preparing Submission

In [ ]:
# Inference + postproc (parametrized, CPU-friendly)

@torch.inference_mode()
def model_probs(model, images, device, use_tta=False):
    model.eval()
    images = images.to(device, non_blocking=True)
    if device.type == "cuda":
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            if use_tta:
                logits = (logits + torch.flip(model(torch.flip(images, dims=[-1])), dims=[-1])) / 2
    else:
        logits = model(images)
        if use_tta:
            logits = (logits + torch.flip(model(torch.flip(images, dims=[-1])), dims=[-1])) / 2
    return torch.sigmoid(logits).cpu().numpy().astype(np.float16)


def cc_filter_mask(mask, min_component):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_component:
            out[labels == i] = 1
    return out


def postprocess(probs, thresholds, min_component, min_pixels):
    # probs (B,C,H,W) float16/float32 -> filtered uint8 masks
    B, C, H, W = probs.shape
    out = np.zeros((B, C, H, W), dtype=np.uint8)
    for c in range(C):
        base = (probs[:, c].astype(np.float32) >= thresholds[c]).astype(np.uint8)
        if min_component[c] > 0 and base.any():
            for b in range(B):
                base[b] = cc_filter_mask(base[b], min_component[c])
        for b in range(B):
            if base[b].sum() >= min_pixels[c]:
                out[b, c] = base[b]
    return out


@torch.inference_mode()
def predict_test(model, test_loader, device, thresholds, min_component, min_pixels, use_tta=False):
    model.eval()
    rle_dict = {}
    pbar = tqdm(test_loader, desc="Generating [Test Preds]", dynamic_ncols=True)
    for images, image_ids in pbar:
        probs = model_probs(model, images, device, use_tta=use_tta)
        preds = postprocess(probs, thresholds, min_component, min_pixels)
        for b_idx, img_id in enumerate(image_ids):
            for c_idx, defect_type in enumerate(DEFECT_TYPES):
                mask = preds[b_idx, c_idx]
                rle_dict[(str(img_id), str(defect_type))] = rle_encode(mask) if mask.sum() > 0 else " "
        pbar.set_postfix({"predictions": len(rle_dict)})
    return rle_dict


In [ ]:
def build_submission(rle_dict, sample_sub_path, output_csv):
    
    sample_sub = pd.read_csv(sample_sub_path)

    sample_sub["mask_rle"] = [ rle_dict.get( (str(row.image_id), str(row.defect_type)), " " ) for row in sample_sub.itertuples(index=False) ]
    sample_sub.to_csv(output_csv, index=False)

    total_rows = len(sample_sub)
    non_empty_count = (sample_sub["mask_rle"] != " ").sum()

    print(f"\nSubmission saved: {output_csv} | Total Rows: {total_rows}")
    print( f"Predicted Defect Masks: {non_empty_count} / {total_rows} | ({non_empty_count / total_rows:.2%})" )

    return sample_sub

In [ ]:
trained_models = {
    "UNET": ["BASELINE", "BASELINE-V1.1"],
    "UNET-RESNET34": ["IMAGENET-FINETUNED-V1.0", ],
    "UNET-EFFICIENTNET-B0": [],
    "RESIDUAL-UNET": ["BASELINE-V1.0"],
    "ATTENTION-UNET": ["BASELINE-V1.0"]
}

tModel = "ATTENTION-UNET"
# Download and load best model
handle = f"coder057/{tModel}/PyTorch/{trained_models[tModel][0].lower()}"
model = load_attached_model(device)

In [ ]:
# ===== Threshold / postproc sweep on validation + submission =====
import time
t_start = time.time()
print("STAGE 1/5 collect validation probs (plain + TTA)", flush=True)

val_probs = {"plain": [], "tta": []}
val_targets = []
for images, targets in tqdm(valid_loader, desc="[Val probs]", dynamic_ncols=True):
    val_targets.append(targets.numpy().astype(np.uint8))
    val_probs["plain"].append(model_probs(model, images, device, use_tta=False))
    val_probs["tta"].append(model_probs(model, images, device, use_tta=True))
val_probs = {k: np.concatenate(v) for k, v in val_probs.items()}
val_targets = np.concatenate(val_targets)
print(f"val probs: {val_probs['plain'].shape}, targets: {val_targets.shape} | {time.time()-t_start:.0f}s elapsed", flush=True)

THRESH_GRID = np.round(np.arange(0.05, 0.801, 0.025), 3)

def competition_score(pred_u8, tgt_u8):
    N, C, H, W = pred_u8.shape
    p = pred_u8.reshape(N, C, H * W).astype(np.float64)
    t = (tgt_u8.reshape(N, C, H * W) > 0).astype(np.float64)
    pred_px, tgt_px = p.sum(-1), t.sum(-1)
    inter = (p * t).sum(-1)
    eps = 1e-7
    pos, emp = tgt_px > 0, tgt_px == 0
    dice = (2 * inter + eps) / (pred_px + tgt_px + eps)
    fg = dice[pos].mean() if pos.any() else 0.0
    fps = 1.0 - (pred_px[emp] / (H * W)).mean() if emp.any() else 1.0
    ea = (pred_px[emp] == 0).mean() if emp.any() else 1.0
    return 0.7 * fg + 0.2 * fps + 0.1 * ea

def best_thresholds(probs, tgt):
    N, C, H, W = probs.shape
    P = probs.reshape(N, C, H * W).astype(np.float32)
    T = (tgt.reshape(N, C, H * W) > 0)
    tgt_px = T.sum(-1).astype(np.float64)
    pos, emp = tgt_px > 0, tgt_px == 0
    best = []
    for c in range(C):
        pc, tc = P[:, c], T[:, c]
        tgt_px_c = tgt_px[:, c]
        pos_c, emp_c = pos[:, c], emp[:, c]
        scores = []
        for thr in THRESH_GRID:
            pred = pc >= thr
            pred_px = pred.sum(-1).astype(np.float64)
            inter = (pred & tc).sum(-1).astype(np.float64)
            dice = (2 * inter + 1e-7) / (pred_px + tgt_px_c + 1e-7)
            fg = dice[pos_c].mean() if pos_c.any() else 0.0
            fps = 1.0 - (pred_px[emp_c] / (H * W)).mean() if emp_c.any() else 1.0
            ea = (pred_px[emp_c] == 0).mean() if emp_c.any() else 1.0
            scores.append(0.7 * fg + 0.2 * fps + 0.1 * ea)
        i = int(np.argmax(scores))
        best.append({"threshold": float(THRESH_GRID[i]), "sub_score": float(scores[i])})
        print(f"  class {DEFECT_TYPES[c]}: best thr={THRESH_GRID[i]:.3f} (score {scores[i]:.4f})", flush=True)
    return best

print("STAGE 2/5 per-class threshold sweep", flush=True)
thresholds = {k: best_thresholds(v, val_targets) for k, v in val_probs.items()}
print(f"{time.time()-t_start:.0f}s elapsed", flush=True)

MC_GRID = [0, 16, 24, 32, 48, 64, 80, 96, 128, 160, 192, 256]
MT_GRID = [0, 64, 128, 192, 256, 320, 384, 512]
rng = np.random.default_rng(CONFIG["seed"])
sub_idx = rng.choice(val_targets.shape[0], size=min(400, val_targets.shape[0]), replace=False)

def sweep_cc(probs, tgt, thr_list):
    results = []
    for c in range(probs.shape[1]):
        thr = thr_list[c]["threshold"]
        P = (probs[sub_idx, c].astype(np.float32) >= thr).astype(np.uint8)
        T = (tgt[sub_idx, c] > 0)
        n = len(sub_idx)
        tgt_px = T.reshape(n, -1).sum(-1).astype(np.float64)
        pos, emp = tgt_px > 0, tgt_px == 0
        T_u8 = T.astype(np.uint8)
        base_sums = P.reshape(n, -1).sum(-1).astype(np.float64)
        base_inter = (P * T_u8).reshape(n, -1).sum(-1).astype(np.float64)
        best = (0, 0, -1.0)
        for mc in MC_GRID:
            if mc == 0:
                sums = base_sums
                itr = base_inter
            else:
                masks = np.stack([cc_filter_mask(P[i], mc) for i in range(n)])
                sums = masks.reshape(n, -1).sum(-1).astype(np.float64)
                itr = (masks * T_u8).reshape(n, -1).sum(-1).astype(np.float64)
            for mt in MT_GRID:
                px = np.where(sums >= mt, sums, 0)
                itr2 = np.where(sums >= mt, itr, 0)
                dice = (2 * itr2 + 1e-7) / (px + tgt_px + 1e-7)
                fg = dice[pos].mean() if pos.any() else 0.0
                fps = 1.0 - (px[emp] / P.shape[-1] / P.shape[-2]).mean() if emp.any() else 1.0
                ea = (px[emp] == 0).mean() if emp.any() else 1.0
                s = 0.7 * fg + 0.2 * fps + 0.1 * ea
                if s > best[2]:
                    best = (mc, mt, s)
        results.append({"min_component": best[0], "min_pixels": best[1], "sub_score": best[2]})
        print(f"  class {DEFECT_TYPES[c]}: mc={best[0]} mt={best[1]} (sub score {best[2]:.4f})", flush=True)
    return results

print("STAGE 3/5 connected-component / min-pixel sweep (400-img subsample)", flush=True)
cc_cfg = {k: sweep_cc(v, val_targets, thresholds[k]) for k, v in val_probs.items()}
print(f"{time.time()-t_start:.0f}s elapsed", flush=True)

print("STAGE 4/5 full-validation evaluation, variant selection", flush=True)
variant_cfg = {}
for k in val_probs:
    thr = [d["threshold"] for d in thresholds[k]]
    mc = [d["min_component"] for d in cc_cfg[k]]
    mt = [d["min_pixels"] for d in cc_cfg[k]]
    pred = np.zeros_like(val_probs[k], dtype=np.uint8)
    for c in range(pred.shape[1]):
        base = (val_probs[k][:, c].astype(np.float32) >= thr[c]).astype(np.uint8)
        if mc[c] > 0:
            for i in range(base.shape[0]):
                base[i] = cc_filter_mask(base[i], mc[c])
        for i in range(base.shape[0]):
            if base[i].sum() >= mt[c]:
                pred[i, c] = base[i]
    sc = competition_score(pred, val_targets)
    variant_cfg[k] = {"thresholds": thr, "min_component": mc, "min_pixels": mt, "val_score": sc}
    print(f"  {k}: val score {sc:.4f} | thr={thr} mc={mc} mt={mt}", flush=True)

baseline_pred = postprocess(val_probs["plain"].astype(np.float32),
                            CONFIG["class_thresholds"],
                            [CONFIG["min_component_pixels"]] * 4,
                            [CONFIG["min_defect_pixels"]] * 4)
print(f"  baseline (old CONFIG, plain): val score {competition_score(baseline_pred, val_targets):.4f}", flush=True)

best_variant = max(variant_cfg, key=lambda k: variant_cfg[k]["val_score"])
cfg = variant_cfg[best_variant]
use_tta = best_variant == "tta"
print(f"BEST variant: {best_variant} | val score {cfg['val_score']:.4f} | {time.time()-t_start:.0f}s elapsed", flush=True)

print(f"STAGE 5/5 test prediction + submission (TTA={use_tta})", flush=True)
rle_dict = predict_test(model, test_loader, device, cfg["thresholds"], cfg["min_component"], cfg["min_pixels"], use_tta=use_tta)
submission_df = build_submission(rle_dict=rle_dict, sample_sub_path=SAMPLE_SUB_PATH, output_csv="/kaggle/working/submission.csv")
display(submission_df.head(12))

# submission validation
sub = pd.read_csv("/kaggle/working/submission.csv")
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
assert list(sub.columns) == list(sample_sub.columns), "column mismatch"
assert len(sub) == len(sample_sub), "row count mismatch"
assert set(sub["image_id"]) == set(sample_sub["image_id"]), "image_id set mismatch"
assert sub.notna().all().all(), "NaN in submission"
print(f"Submission validation PASSED | total {time.time()-t_start:.0f}s", flush=True)
